In [34]:
# need to restart jupyter server? kernal? whenever a change is made to the pybamm module? 
import sys
import os
import numpy as np
import pandas as pd
import os
from scipy import integrate
import matplotlib.pyplot as plt
sys.path.insert(0, r'c:\Users\Vivian\Dropbox (University of Michigan)\from_box\Research\PyBaMM\PyBaMM')
import pybamm
print(pybamm.__path__[0])
%matplotlib widget
# inline
import inspect
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': 'Computer Modern',
    'font.sans-serif': ['Computer Modern'],
    'font.size': 11,
})


c:\Users\Vivian\Dropbox (University of Michigan)\from_box\Research\PyBaMM\PyBaMM\pybamm


In [35]:
class ExternalCircuitResistanceFunction():
    def __call__(self, variables):
        I = variables["Current [A]"]
        V = variables["Terminal voltage [V]"]
        R_ext = pybamm.FunctionParameter("External short resistance [Ohm]",  {"Time [s]": pybamm.t}) 
        R_tab = pybamm.FunctionParameter("Tabbing resistance [Ohm]",  {"Time [s]": pybamm.t})
        return V/I - (R_ext + R_tab)

# Set up concentration dependent functions

In [36]:
from pybamm import exp, constants, Parameter


def modified_graphite_diffusivity_PeymanMPM(sto, T):
    D_ref =  8e-14 #Parameter("Negative electrode diffusion coefficient [m2.s-1]")
    E_D_s = 42770/10
    soc = (sto - 0)/(0.8321-0)
    k = 1.12070451*soc + 0.09209274 # Ds_restart_rmseV_11_simultaneous_rest (updated exp C-rate) only exclude kn>9
    return D_ref*k # *(-0.9 * sto + 1)

def modified_NMC_diffusivity_PeymanMPM(sto, T):
    D_ref =  8e-15 #Parameter("Positive electrode diffusion coefficient [m2.s-1]")
    E_D_s = 18550/10
    soc = (0.837-sto)/(0.837-0.034)
    k =  4.7302281*soc**2  -4.5023245*soc + 1.26466141 # # Ds_restart_rmseV_11_simultaneous_rest (updated exp C-rate)
    return D_ref *k

def modified_electrolyte_diffusivity_PeymanMPM(c_e, T):
    D_c_e =  Parameter("Typical electrolyte diffusivity [m2.s-1]")
    E_D_e = 37040
    arrhenius = exp(E_D_e / constants.R * (1 / 298.15 - 1 / T))
    return D_c_e#*arrhenius

def modified_electrolyte_conductivity_PeymanMPM(c_e, T):
    # sigma_e = 1.3
    sigma_e = Parameter("Typical electrolyte conductivity [m2.s-1]")
    E_k_e = 34700
    return sigma_e#*k_T# (1-k)#*arrhenius

def modified_NMC_electrolyte_exchange_current_density_PeymanMPM(c_e, c_s_surf, c_s_max, T):
    m_ref =  Parameter("Positive electrode reference exchange-current density [A.m-2(m3.mol)1.5]")
    E_r = 39570
    return (m_ref * c_e**0.5 * c_s_surf**0.5 * (c_s_max - c_s_surf) ** 0.5)

def modified_graphite_electrolyte_exchange_current_density_PeymanMPM(c_e, c_s_surf, c_s_max, T):
    m_ref =  Parameter("Negative electrode reference exchange-current density [A.m-2(m3.mol)1.5]")
    E_r = 37480
    arrhenius = exp(E_r / constants.R * (1 / 298.15 - 1 / T))

    return (
        m_ref * c_e**0.5 * c_s_surf**0.5 * (c_s_max - c_s_surf) ** 0.5
    )

# Simulate 100% SOC ESC

## Nominal model

In [37]:
SOC_0 = 1
SOC_name = SOC_0
Q_nom = 4.6
h = 16.58819844 #100% dT loss function
Cp = 1.1797067 
R_tab = pybamm.Parameter("Tabbing resistance [Ohm]")
R_ext = pybamm.Parameter("External resistance [Ohm]")
data_sets = []

# load data import ESC data from file
raw_data = pd.read_csv("ESC_"+  "100A" + "SOC_full.csv")
esc_start = raw_data[-raw_data['Current Shunt']>1].index[0]
esc_end = len(raw_data)
# T_amb = np.mean(raw_data['Cell Temperature'][(raw_data.index < esc_start) & (raw_data['Cell Temperature'] >0)])
data = raw_data[['Time (s)', 'Voltage (V)', 'Cell Temperature', 'Current Shunt', 'Force']].loc[esc_start:esc_end].copy()
data['Current Shunt'] = -data['Current Shunt']
data['Time (s)'] = data['Time (s)'] - data['Time (s)'].loc[esc_start]
df_labels = ['t', 'V', 'Temp','I', 'F']
data.set_axis(df_labels, axis=1, inplace=True)
data['I_C'] = data.I/Q_nom 
AhT_calculated = integrate.cumtrapz(abs(data.I), data.t-data.t.iloc[0])/3600
AhT_calculated = np.append(AhT_calculated,AhT_calculated[-1])
data['SOC'] = SOC_0 - AhT_calculated/Q_nom 
T_amb = data.Temp.iloc[0]
data_sets.append(data)

options = {
    "thermal": "lumped",
    "decomposition": "true", 
    "operating mode": ExternalCircuitResistanceFunction(),
}
model = pybamm.lithium_ion.SPMe(options = options)
chemistry = pybamm.parameter_sets.Tran2023
param = pybamm.ParameterValues(chemistry)
param.update({

    "Tabbing resistance [Ohm]":  0.0086,
    "External short resistance [Ohm]": 0.0067, 
    "Cell capacity [A.h]": Q_nom, #nominal
    "Typical current [A]": Q_nom,
    'Nominal cell capacity [A.h]': Q_nom,
    "Negative electrode thickness [m]":62E-06*4.2/5,
    "Positive electrode thickness [m]":67E-06*4.2/5,
    "Lower voltage cut-off [V]": 0,
    "Ambient temperature [K]":T_amb + 273.15,
    "Initial temperature [K]": T_amb + 273.15,
    "Negative electrode diffusivity [m2.s-1]": modified_graphite_diffusivity_PeymanMPM,
    "Positive electrode diffusivity [m2.s-1]": modified_NMC_diffusivity_PeymanMPM,
    "Electrolyte diffusivity [m2.s-1]": modified_electrolyte_diffusivity_PeymanMPM,
    "Electrolyte conductivity [S.m-1]": modified_electrolyte_conductivity_PeymanMPM,
    # "Initial concentration in electrolyte [mol.m-3]":1000, #*1.3
    "Negative electrode exchange-current density [A.m-2]": modified_graphite_electrolyte_exchange_current_density_PeymanMPM,
    "Positive electrode exchange-current density [A.m-2]": modified_NMC_electrolyte_exchange_current_density_PeymanMPM,
    "Negative electrode OCP entropic change [V.K-1]":0,
    "Positive electrode OCP entropic change [V.K-1]":0,

}, check_already_exists = False)

V = model.variables["Terminal voltage [V]"]
I = model.variables["Current [A]"]
model.variables.update({
    "Terminal voltage [V]": V - I*R_tab,
    "Actual resistance [Ohm]":V/I,
    }
)

dt = 0.1
t_eval = np.arange(0, 5*60, dt)
solver = pybamm.CasadiSolver(mode="safe", dt_max=1) #, extra_options_setup={"max_num_steps": 10000}
sim = pybamm.Simulation(model, parameter_values = param, solver=solver) 
solution = sim.solve( initial_soc=SOC_0, t_eval = t_eval)


In [38]:
sim.var_pts

{'x_n': 20,
 'x_s': 20,
 'x_p': 20,
 'r_n': 20,
 'r_p': 20,
 'r_n_prim': 20,
 'r_p_prim': 20,
 'r_n_sec': 20,
 'r_p_sec': 20,
 'y': 10,
 'z': 10,
 'R_n': 30,
 'R_p': 30}

In [68]:
test = solution['Negative particle concentration'].entries[0][0][0::]
print(test)
print(test.shape)


[0.83337428 0.83337428 0.83337428 ... 0.43707563 0.43697326 0.43687094]
(3000,)


In [47]:
%matplotlib inline
quick_plot = pybamm.QuickPlot(solution, figsize=[10,4],n_rows = 2, output_variables=[ 
    'Terminal voltage [V]',
    'X-averaged negative particle surface concentration',
    'X-averaged positive particle surface concentration',

    'Volume-averaged cell temperature [K]',
    'Negative particle concentration', 
    'X-averaged positive particle concentration',

    'Negative electrolyte concentration',
    'Positive electrolyte concentration',
    ])
quick_plot.dynamic_plot()

interactive(children=(FloatSlider(value=0.0, description='t', max=299.90000000000003, step=2.9990000000000006)…

In [41]:
def _heat_of_mixing(self, variables):
    """Compute heat of mixing source terms."""
    param = self.param

    if self.options["heat of mixing"] == "true":
        F = pybamm.constants.F.value
        pi = np.pi

        # Compute heat of mixing in negative electrode
        # if self.options.electrode_types["negative"] == "planar":
        #     Q_mix_s_n = pybamm.FullBroadcast(
        #         0, ["negative electrode"], "current collector"
        #     )
        # else:
        a_n = variables["Negative electrode surface area to volume ratio [m-1]"]
        R_n = variables["Negative particle radius [m]"]
        N_n = a_n / (4 * pi * R_n**2)
        c_n = variables["Negative particle concentration [mol.m-3]"]
        T_n = variables["Negative electrode temperature [K]"]
        
        T_n_part = pybamm.PrimaryBroadcast(T_n, ["negative particle"])
        # c_n_part = pybamm.FullBroadcast(c_n , ["negative particle"], 'current collector')
        D_n = param.n.prim.D(c_n, T_n_part)
        gc = pybamm.grad(c_n)
        dc_n_dr2 = pybamm.inner(gc,gc)
        dUeq_n = param.n.prim.dUdsto_dimensional(c_n / param.n.prim.c_max, T_n_part)
        integrand_r_n = D_n * dc_n_dr2 * dUeq_n/ param.n.prim.c_max # domain: negative particle
        integration_variable_r_n = pybamm.SpatialVariable("r", domain=integrand_r_n.domain)
        # integration_variable_r_n = [
        #     pybamm.SpatialVariable("r", domain=["negative particle"])#, coord_sys="spherical polar"), 
        # ]
        # integration_variable_r_n = pybamm.SpatialVariable(
        #         "r_n",
        #         domain=["negative particle"],
        #         auxiliary_domains={
        #             "secondary": "negative electrode",
        #             "tertiary": "current collector",
        #         },
        #         # coord_sys="cartesian",
        #     )

        integral_r_n = pybamm.Integral(integrand_r_n, integration_variable_r_n) # domain: negative electrode
        Q_mix_s_n = -F * N_n * integral_r_n # domain: negative electrode
        # Q_mix_s_n = pybamm.FullBroadcast(
        #     5, ["negative electrode"], "current collector"
        # )

        # # Compute heat of mixing in positive electrode
        # a_p = variables["Positive electrode surface area to volume ratio [m-1]"]
        # R_p = variables["Positive particle radius [m]"]
        # N_p = a_p / (4 * pi * R_p**2)
        # # if self.x_average:
        # #     c_p = variables["X-averaged positive particle concentration [mol.m-3]"]
        # #     T_p = variables["X-averaged positive electrode temperature [K]"]
        # # else:
        # c_p = variables["Positive particle concentration [mol.m-3]"]
        # T_p = variables["Positive electrode temperature [K]"]
        # T_p_part = pybamm.PrimaryBroadcast(T_p, ["positive particle"])
        # dc_p_dr2 = pybamm.inner(pybamm.grad(c_p), pybamm.grad(c_p))
        # D_p = param.p.prim.D(c_p, T_p_part)
        # dUeq_p = param.p.prim.dUdsto_dimensional(c_p / param.p.prim.c_max, T_p_part)
        # integrand_r_p = D_p * dc_p_dr2 * dUeq_p / param.p.prim.c_max
        # integration_variable_r_p = [
        #     pybamm.SpatialVariable("r", domain=integrand_r_p.domain)
        # ]
        # integral_r_p = pybamm.Integral(integrand_r_p, integration_variable_r_p)
        # Q_mix_s_p = -F * N_p * integral_r_p
        # Q_mix_s_s = pybamm.FullBroadcast(0, ["separator"], "current collector")
    else:
        Q_mix_s_n = pybamm.FullBroadcast(
            0, ["negative electrode"], "current collector"
        )
    Q_mix_s_p = pybamm.FullBroadcast(
        0, ["positive electrode"], "current collector"
    )
    Q_mix_s_s = pybamm.FullBroadcast(
        0, ["separator"], "current collector"
    )

    return Q_mix_s_n, Q_mix_s_s, Q_mix_s_p

In [42]:
def sim_thermal(t, Q):
    chemistry = pybamm.parameter_sets.Tran2023
    param = pybamm.ParameterValues(chemistry)

    param.update({
        "Ambient temperature [K]":25 + 273.15,
        "Initial temperature [K]": 25 + 273.15,
        "Negative tab width [m]":2.5e-2,
        "Positive tab width [m]":2.5e-2,
        "Total heat transfer coefficient [W.m-2.K-1]":h,
        "Negative electrode specific heat capacity [J.kg-1.K-1]": 1100*Cp,
        "Positive electrode specific heat capacity [J.kg-1.K-1]": 1100*Cp,
        "Cell capacity [A.h]": 4.6, #nominal
        "Typical current [A]": 4.6,
        "Negative electrode thickness [m]":62E-06*4.2/5,
        "Positive electrode thickness [m]":67E-06*4.2/5,
    }, check_already_exists = False)

    # Define input and output
    Q_vol_av = pybamm.Variable("Volume-averaged total heating")
    T_vol_av= pybamm.Variable("Volume-averaged cell temperature")

    # Init model and variables
    liion = pybamm.LithiumIonParameters()
    model = pybamm.BaseModel()
    Delta_T = param.evaluate(liion.Delta_T)
    T_ref = param.evaluate(liion.T_ref)
    tau_discharge = param.evaluate(liion.tau_discharge)
    tau_th_yz = param.evaluate(liion.tau_th_yz)
    model.timescale = pybamm.Scalar(tau_discharge)
    model.variables = {
        "Volume-averaged total heating": Q_vol_av, 
        "Volume-averaged cell temperature": T_vol_av,
        "Volume-averaged cell temperature [K]": Delta_T * T_vol_av + T_ref,
    }
    model.external_variables = [Q_vol_av]

    # Set up parameters
    delta = param.evaluate(liion.delta) #self.L_x / self.L_z
    cell_surface_area = param.evaluate(liion.a_cooling) #self.A_cooling / (self.L_z ** 2)
    cell_volume = param.evaluate(liion.v_cell) # self.V_cell / (self.L_x * self.L_z ** 2)
    h_total = param.evaluate(liion.h_total) # self.h_total_dim * self.geo.L_x / self.lambda_eff_dim_ref
    rho = param.evaluate(liion.rho(T_vol_av)) # Dimensionless effective density? self.rho_dim(T_dim) * self.c_p_dim(T_dim) / self.main_param.rho_eff_dim_ref
    T_amb = param.evaluate(liion.T_amb(0)) # (self.T_amb_dim(t) - self.T_ref) / self.Delta_T
    total_cooling_coefficient = (
        -h_total
        * cell_surface_area
        / cell_volume
        / (delta ** 2))
    B = param.evaluate(liion.B) #self.B = (self.i_typ* self.R* self.T_ref* self.tau_th_yz/ (self.therm.rho_eff_dim_ref * self.F * self.Delta_T * self.L_x))
    C_th = param.evaluate(liion.tau_th_yz)/param.evaluate(liion.tau_discharge) # self.tau_th_yz / self.timescale
    Q_scale = param.evaluate(liion.i_typ) * param.evaluate(liion.potential_scale) / param.evaluate(liion.L_x)


    model.variables.update({
    "Volume-averaged total heating [W.m-3]": Q_vol_av * Q_scale,
    })

    # RHS
    model.rhs = {
        T_vol_av: (
            B * Q_vol_av + total_cooling_coefficient * (T_vol_av - T_amb)
        )/ (C_th * rho)
    }

    # set initial conditions
    T_init = param.evaluate(liion.T_init)
    model.initial_conditions = {T_vol_av: pybamm.Scalar(T_init)}

    # discretize
    disc = pybamm.Discretisation()  # use the default discretisation
    disc.process_model(model)

    # solve
    sim = pybamm.Simulation(model)
    for (dt,q) in zip(np.diff(t), Q[0:-1]):
        external_variables = {"Volume-averaged total heating": q/Q_scale} 
        sim.step(dt, external_variables=external_variables)
    return sim.solution

# t = solution['Time [s]'].entries
# Q = solution['Volume-averaged total heating [W.m-3]'](t)
# solution_test = sim_thermal(t,Q)
# pybamm.dynamic_plot(
#     solution_test,
#     output_variables=[
#         "Volume-averaged total heating [W.m-3]",
#         "Volume-averaged cell temperature [K]"
#     ],
# )


# Test $Q_{mix}$ model

In [43]:
chemistry = pybamm.parameter_sets.Tran2023
param = pybamm.ParameterValues(chemistry)

param.update({
    "Ambient temperature [K]":25 + 273.15,
    "Initial temperature [K]": 25 + 273.15,
    "Negative tab width [m]":2.5e-2,
    "Positive tab width [m]":2.5e-2,
    "Total heat transfer coefficient [W.m-2.K-1]":h,
    "Negative electrode specific heat capacity [J.kg-1.K-1]": 1100*Cp,
    "Positive electrode specific heat capacity [J.kg-1.K-1]": 1100*Cp,
    "Cell capacity [A.h]": 4.6, #nominal
    "Typical current [A]": 4.6,
    "Negative electrode thickness [m]":62E-06*4.2/5,
    "Positive electrode thickness [m]":67E-06*4.2/5,
}, check_already_exists = False)

In [44]:
a_n = solution["Negative electrode surface area to volume ratio [m-1]"]
R_n = solution["Negative particle radius [m]"]

In [45]:
solution["Negative particle concentration [mol.m-3]"]

In [46]:
# def sim_Q_mix(solution):
    # Define input and output
Q_mix = pybamm.Variable("Heat of mixing [W.m-3]")
a_n = pybamm.Variable("Negative electrode surface area to volume ratio [m-1]")
R_n =pybamm.Variable("Negative particle radius [m]")
c_n = solution["Negative particle concentration [mol.m-3]"] #pybamm.Variable("Negative particle concentration [mol.m-3]")
T_n = pybamm.Variable("Negative electrode temperature [K]")# Init model and variables

liion = pybamm.LithiumIonParameters()
model = pybamm.BaseModel()
model.timescale = pybamm.Scalar(param.evaluate(liion.tau_discharge))
model.variables = {
    "Heat of mixing [W.m-3]": Q_mix, 
    "Negative electrode surface area to volume ratio [m-1]": a_n,
    "Negative particle radius [m]": R_n,
    "Negative particle concentration [mol.m-3]": c_n,
    "Negative electrode temperature [K]":T_n
}
# model.external_variables = [a_n, R_n, c_n, T_n]

# Set up parameters
F = pybamm.constants.F.value
pi = np.pi
N_n = a_n / (4 * pi * R_n**2)
T_n_part = pybamm.PrimaryBroadcast(T_n, ["negative particle"])
dc_n_dr2 = pybamm.inner(pybamm.grad(c_n), pybamm.grad(c_n))
D_n = param.n.prim.D(c_n, T_n_part)
dUeq_n = param.n.prim.dUdsto_dimensional(c_n / param.n.prim.c_max, T_n_part)
integrand_r_n = D_n * dc_n_dr2 * dUeq_n / param.n.prim.c_max # domain: negative particle
integration_variable_r_n = [
    pybamm.SpatialVariable("r", domain=integrand_r_n.domain)
]
integral_r_n = pybamm.Integral(integrand_r_n, integration_variable_r_n) # domain: negative electrode
Q_mix_s_n = -F * N_n * integral_r_n # domain: negative electrode
Q_scale = param.evaluate(liion.i_typ) * param.evaluate(liion.potential_scale) / param.evaluate(liion.L_x)

model.variables.update({
"Heat of mixing [W.m-3]": Q_mix * Q_scale,
})


# discretize
disc = pybamm.Discretisation()  # use the default discretisation
disc.process_model(model)

# # solve
# sim = pybamm.Simulation(model)
# for (dt,q) in zip(np.diff(t), Q[0:-1]):
#     external_variables = {"Volume-averaged total heating": q/Q_scale} 
#     sim.step(dt, external_variables=external_variables)
# return sim.solution

AttributeError: 'ProcessedVariable' object has no attribute 'evaluates_on_edges'

In [ ]:
# param.evaluate(liion.n.prim.dUdsto_dimensional(0.5, 298))
# param.evaluate(liion.rho(298))
sto = 0.5
Domain = 'Negative'
domain =Domain.lower
tol = pybamm.settings.tolerances["U__c_s"]
sto = pybamm.maximum(pybamm.minimum(sto, 1 - tol), tol)
u_ref = pybamm.FunctionParameter(
    "Negative electrode OCP [V]",
    {"Negative particle stoichiometry": sto},
    diff_variable=sto,
)

dudt = pybamm.FunctionParameter(
    "Negative electrode OCP entropic change [V.K-1]",
    {
        "Negative particle stoichiometry": sto,
        "Maximum negative particle surface concentration [mol.m-3]": liion.n.prim.c_max,
    },
    diff_variable=sto,
)

u_ref + (298 - param.evaluate(liion.T_ref)) * dudt

Addition(-0x2355e683f694edd1, +, children=['Negative electrode OCP [V]', '-0.14999999999997726 * Negative electrode OCP entropic change [V.K-1]'], domains={})

In [ ]:
model.variables.search('temperature')

Ambient temperature
Ambient temperature [K]
Cell temperature
Cell temperature [K]
Negative current collector temperature
Negative current collector temperature [K]
Negative electrode temperature
Negative electrode temperature [K]
Positive current collector temperature
Positive current collector temperature [K]
Positive electrode temperature
Positive electrode temperature [K]
Separator temperature
Separator temperature [K]
Volume-averaged cell temperature
Volume-averaged cell temperature [K]
X-averaged cell temperature
X-averaged cell temperature [K]
X-averaged negative electrode temperature
X-averaged negative electrode temperature [K]
X-averaged positive electrode temperature
X-averaged positive electrode temperature [K]
X-averaged separator temperature
X-averaged separator temperature [K]


In [ ]:
chemistry = pybamm.parameter_sets.Tran2023
param = pybamm.ParameterValues(chemistry)
liion = pybamm.LithiumIonParameters()
F = pybamm.constants.F.value
pi = np.pi

variables = solution
sim_time = solution['Time [s]'].entries
x = solution['x [m]'].entries
r_n = variables['r_n [m]'].entries
Q_mix_s_n=[]
for t in sim_time[0:1]:
    c_n = variables["Negative particle concentration [mol.m-3]"](t=t, r = r_n, x=x)
    # T_n = variables["X-averaged negative electrode temperature [K]"](t=t, x=x)
    # a_n = variables["Negative electrode surface area to volume ratio [m-1]"](t=t, r = r_n, x=x)
    # N_n = a_n / (4 * pi * r_n**2)
    # T_n_part = pybamm.PrimaryBroadcast(T_n, ["negative particle"])
    # dc_n_dr2 = pybamm.inner(pybamm.grad(c_n), pybamm.grad(c_n))
    # D_n = liion.n.prim.D(c_n, T_n_part)
    # dUeq_n = liion.n.prim.dUdsto_dimensional(c_n / liion.n.prim.c_max, T_n_part)
    # integrand_r_n = D_n * dc_n_dr2 * dUeq_n / liion.n.prim.c_max
    # integration_variable_r_n = [
    #     pybamm.SpatialVariable("r", domain=integrand_r_n.domain)
    # ]
    # integral_r_n = pybamm.Integral(integrand_r_n, integration_variable_r_n)
    # Q_mix_s_n.append(F * N_n * integral_r_n)
